In [1]:
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim
import re
import math

# Data Cleaning And Tokenization

In [2]:
text = """
i love artificial intelligence and machine learning.
artificial intelligence is transforming the world.
machine learning allows computers to learn from data.
deep learning is a part of machine learning.
ai is powerful and exciting.
"""

# text = text.split() better way but now ew are working in paragraph

In [3]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '.', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x']


In [4]:
print(f"Total Unque Vocab {len(chars)}")

Total Unque Vocab 23


In [5]:
word_to_int = {}

int_to_word = {}

for i,data in enumerate(chars):
    word_to_int[data] = i


for i,data in enumerate(chars):
    int_to_word[i] = data

    

In [6]:
word_to_int

{'\n': 0,
 ' ': 1,
 '.': 2,
 'a': 3,
 'c': 4,
 'd': 5,
 'e': 6,
 'f': 7,
 'g': 8,
 'h': 9,
 'i': 10,
 'l': 11,
 'm': 12,
 'n': 13,
 'o': 14,
 'p': 15,
 'r': 16,
 's': 17,
 't': 18,
 'u': 19,
 'v': 20,
 'w': 21,
 'x': 22}

In [7]:
int_to_word

{0: '\n',
 1: ' ',
 2: '.',
 3: 'a',
 4: 'c',
 5: 'd',
 6: 'e',
 7: 'f',
 8: 'g',
 9: 'h',
 10: 'i',
 11: 'l',
 12: 'm',
 13: 'n',
 14: 'o',
 15: 'p',
 16: 'r',
 17: 's',
 18: 't',
 19: 'u',
 20: 'v',
 21: 'w',
 22: 'x'}

# Encoding and decoding (Testing)

In [8]:
def encoder(data):
    result = []
    for char in data:
         result.append(word_to_int[char])
    return result
    

def decoder(data):
    result = " "
    for char in data:
        result+=int_to_word[char]

    return result
    

In [9]:
# Testing

encoded_value = encoder("deep learning is a part of machine learning.")
print(encoded_value)

[5, 6, 6, 15, 1, 11, 6, 3, 16, 13, 10, 13, 8, 1, 10, 17, 1, 3, 1, 15, 3, 16, 18, 1, 14, 7, 1, 12, 3, 4, 9, 10, 13, 6, 1, 11, 6, 3, 16, 13, 10, 13, 8, 2]


In [10]:
# Decoding 
decoded_value = decoder(encoded_value)
print(decoded_value)

 deep learning is a part of machine learning.


# Embadding values

In [11]:
def embadding(input_tensor, vocab_size, d_model=5):
    embedding = nn.Embedding(vocab_size, d_model)
    return embedding(input_tensor)

In [12]:
vocab_size = len(chars)
encoded = encoder(text)
input_tensor = torch.tensor(encoded).unsqueeze(0)

embedded = embadding(input_tensor, vocab_size)

print(embedded.shape)
print(embedded)

torch.Size([1, 233, 5])
tensor([[[ 0.3048,  0.9858, -0.8615, -0.3799, -0.1409],
         [-1.2086,  1.3913, -0.0148, -0.5146,  0.3456],
         [ 1.6625,  0.1688, -0.3148, -0.8036,  0.2748],
         ...,
         [ 0.9414, -1.3797,  0.2290,  2.0024, -2.0649],
         [ 1.0566, -0.5610, -0.1754, -1.0749,  0.5978],
         [ 0.3048,  0.9858, -0.8615, -0.3799, -0.1409]]],
       grad_fn=<EmbeddingBackward0>)


# Postional Encodeere

In [13]:
# When we convert words into embeddings (numbers), we lose the order of words.

# Example:

# "lion eats human"
# "human eats lion"
# "eat lion human"

# All contain same words, but meaning is different

 # Problem

# Embeddings only capture:

# meaning of words  but NOT their position 

 # So model cannot understand:

# who is doing action
# who is receiving action
#  Solution: Positional Encoding

# To solve this, we add extra information about position.

#  This is done using:

# sine (sin)
# cosine (cos) functions

# with different frequencies.


# Affect after adding Positional encoder 
#  no of dimension  of postional  == no of embadding dimension  
#  Small changes in position lead to smooth and consistent changes in the encoding vectors, which helps the model learn relationships between nearby words effectively.

In [14]:
# Workflow

# We have a sentence: "hello world"

# Step 1: Embedding
# hello → [0.0, 0.1, 0.9]
# world → [0.1, 0.5, 1.0]

# Step 2: Positional Encoding
# We DO NOT apply sin and cos to embedding values.

# Instead, we generate positional vectors using sin and cos based on position index:

# pos 0 → [sin(0), cos(0), ...]
# pos 1 → [sin(1), cos(1), ...]

# Step 3: Add them
# hello_final = embedding(hello) + PE(pos=0)
# world_final = embedding(world) + PE(pos=1)

# So, number of embeddings = number of positions (sequence length)

# Each word gets:
# 1 embedding vector + 1 positional encoding vector

# Embedding = what the word is
# Position = where the word is
# Final = what + where

In [15]:
def positional_embad(seq_len,embadded_dim):  #seq len give total sentence contain like hello world  = seq_len = 2
    pe = torch.zeros(seq_len,embadded_dim)

    for pos in range(seq_len):

        for i in range(0,embadded_dim,2):  #genreate even number
            angle = pos / (10000 ** (i / embadded_dim))
            # Now assigning Sin in even and cos in odd
            
            pe[pos,i] = math.sin(angle)

            # so evert time odd vlaue or index may not exist
            if i+1 <embadded_dim:
                pe[pos,i+1] = math.cos(angle)


    return pe
                
            

In [16]:
# debug + Test
embedded.shape

torch.Size([1, 233, 5])

In [17]:
# Now cancatting embadded + Position

seq_len = embedded.shape[1]  #total length in 1 sentence
embedded_dim =embedded.shape[2]  


position_emb = positional_embad(seq_len,embedded_dim)

position_emb




tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  2.5116e-02,  9.9968e-01,  6.3096e-04],
        [ 9.0930e-01, -4.1615e-01,  5.0217e-02,  9.9874e-01,  1.2619e-03],
        ...,
        [-6.1606e-01, -7.8770e-01, -4.8455e-01,  8.7476e-01,  1.4461e-01],
        [-9.9568e-01,  9.2806e-02, -4.6242e-01,  8.8666e-01,  1.4524e-01],
        [-4.5988e-01,  8.8798e-01, -4.4001e-01,  8.9799e-01,  1.4586e-01]])

In [18]:
final_embadded = embedded + position_emb

In [19]:
print(f"Dim is Embadded value {embedded.shape}")
print()
print(f"Positional Embadded value {position_emb.shape}")
print()
print(f"Embadded + Positional value : {final_embadded.shape}")

Dim is Embadded value torch.Size([1, 233, 5])

Positional Embadded value torch.Size([233, 5])

Embadded + Positional value : torch.Size([1, 233, 5])


In [20]:
final_embadded

tensor([[[ 0.3048,  1.9858, -0.8615,  0.6201, -0.1409],
         [-0.3672,  1.9316,  0.0103,  0.4851,  0.3463],
         [ 2.5718, -0.2473, -0.2646,  0.1951,  0.2760],
         ...,
         [ 0.3253, -2.1674, -0.2555,  2.8772, -1.9203],
         [ 0.0609, -0.4682, -0.6379, -0.1882,  0.7430],
         [-0.1551,  1.8737, -1.3015,  0.5181,  0.0050]]],
       grad_fn=<AddBackward0>)


# Now Self Attention

In [21]:
def self_attention(embadded_dim,final_embadded):

    wq = nn.Linear(embadded_dim,embadded_dim)
    wk = nn.Linear(embadded_dim,embadded_dim)
    wv = nn.Linear(embadded_dim,embadded_dim)
    
    
    q_matrix = wq(final_embadded)
    k_matrix = wk(final_embadded)
    v_matrix= wv(final_embadded)


    # Now qkT
    kT = k_matrix.transpose(-2,-1)

    # Scaling with Dot Product to reduct large value cause large value and when large value feed to softmax we get value close to 1 or 0 enf to end value

    scores = q_matrix@kT

    # now scaling
    scores = scores/math.sqrt(embadded_dim)


    attention_wts = torch.softmax(scores,dim=-1)  #this provide probability range which say how much to focus on each word
    
    # now 
    output = attention_wts@v_matrix

    return output
    

In [22]:
self_at = self_attention(embedded_dim,final_embadded)
self_at

tensor([[[ 0.4134,  0.2273, -0.0905, -0.5629,  0.2510],
         [ 0.4645,  0.0572, -0.0592, -0.4646,  0.1608],
         [ 0.5009,  0.1822, -0.1611, -0.4599,  0.3303],
         ...,
         [ 0.2733,  0.2774, -0.2004, -0.4919, -0.1044],
         [ 0.4944, -0.0066, -0.1036, -0.3837,  0.0506],
         [ 0.3955,  0.2311, -0.0853, -0.5738,  0.2091]]],
       grad_fn=<UnsafeViewBackward0>)

In [23]:
self_at.shape

torch.Size([1, 233, 5])

# Multi- Head Attention

In [24]:
# Multi head atteintion is same as Self attention but  different is for single attention we called single head attaention
# for multiple attention we call it a Multi head Attention


In [25]:
# Debug
print(embedded_dim)
print(final_embadded.shape)

5
torch.Size([1, 233, 5])


In [26]:
# head_dim	features per head
# num_heads	number of splits
def multi_head_attention(embadded_dim,final_embadded,num_heads):
    head_dim = embadded_dim // num_heads
    # wq = torch.arange(embadded_dim,embadded_dim)
    if embadded_dim % num_heads == 0:
        wq = nn.Linear(embadded_dim,embadded_dim)  #cause embadded and and final have same dim so can use any
        wk = nn.Linear(embadded_dim,embadded_dim)
        wv = nn.Linear(embadded_dim,embadded_dim)

        q = wq(final_embadded)
        k = wk(final_embadded)
        v = wv(final_embadded)


        batch_size,seq_len,embadded_dim = final_embadded.shape

        # splitting embadded_dim into num_head and head_dim
        q = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)  # seq tokens each token now has num_head heads each head has head_dim features
        k = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)  #(batch, seq_len, num_heads, head_dim) -------> (batch, num_heads, seq_len, head_dim)
        v = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)


        kT = k.transpose(-2,-1)

        # scores = q @ kT
        scores = q @ kT
        scores = scores / math.sqrt(head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous() #(batch, num_heads, seq_len, head_dim) -----> (batch, seq_len, num_heads, head_dim)  #contiguous mean data is stored in continious memory
        out = out.view(batch_size, seq_len, embadded_dim)

        return out

    else:
        print("Num head and nead dim embadded_dim % num_heads == 0:")

        

In [27]:
multihead_at = multi_head_attention(embedded_dim,final_embadded,1)
multihead_at

tensor([[[-0.7045,  0.1593,  0.0095,  0.1507, -0.1242],
         [-0.6673,  0.3231, -0.0910,  0.1184, -0.1154],
         [-0.7095, -0.1452,  0.0211,  0.1638, -0.0205],
         ...,
         [-0.0979, -0.1393, -0.4329, -0.1629,  0.2129],
         [-0.3001,  0.1450, -0.3041, -0.0693,  0.0562],
         [-0.6071,  0.2056, -0.0474,  0.0954, -0.1101]]],
       grad_fn=<ViewBackward0>)

# Add and Normalization

In [28]:
def add_norm(final_emb, attention_output, embadded_dim):

    norm = nn.LayerNorm(embadded_dim)

    x = final_emb + attention_output # residual connection  x skip connection  att_op op from attention
    x = norm(x)               # normalization

    return x

In [29]:
x = add_norm(final_embadded,multihead_at, embedded_dim)
x

tensor([[[-0.6332,  1.7379, -1.0544,  0.4574, -0.5077],
         [-1.3272,  1.7271, -0.4415,  0.1938, -0.1522],
         [ 1.8683, -0.9511, -0.7648, -0.0115, -0.1408],
         ...,
         [ 0.3291, -1.1098, -0.1909,  1.7411, -0.7695],
         [-0.0834, -0.2333, -1.3380, -0.1160,  1.7706],
         [-0.7214,  1.6689, -1.2148,  0.4359, -0.1686]]],
       grad_fn=<NativeLayerNormBackward0>)


# Feed Forward NN

In [30]:
def ff_neuralnetwork(x,embadded_dim):
    model = nn.Sequential(nn.Linear(embadded_dim,65),
                          nn.ReLU(),
                          nn.Linear(65,embadded_dim))
    return model(x)
    

In [31]:
ff_nn = ff_neuralnetwork(x,embedded_dim)
ff_nn

tensor([[[ 0.1680,  0.2475, -0.3371, -0.3302, -0.1159],
         [ 0.0156,  0.0697, -0.5188, -0.2153, -0.0644],
         [-0.2555,  0.1514,  0.2799, -0.0860,  0.3280],
         ...,
         [ 0.1438,  0.0735, -0.1196, -0.1778,  0.1199],
         [-0.3598,  0.1334, -0.1454, -0.1004,  0.2504],
         [ 0.1308,  0.2324, -0.3736, -0.2955, -0.1125]]],
       grad_fn=<ViewBackward0>)

# Decoder Architecture Block

In [32]:
# The decoder predicts the next token using only previously seen tokens.
# Masking ensures that future tokens are hidden, preventing information leakage and making training consistent with real-world generation.

# Masked Attention

In [33]:
def masked_multihead_attention(seq_length):  #total lenght of word
    masked = torch.tril(torch.ones(seq_length,seq_length))
    return masked

In [41]:
# Testing
tt = torch.tril(torch.ones(3,3))
print(tt)

# hiding actual data 

print(tt.masked_fill(tt == 0,float('-inf')))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[1., -inf, -inf],
        [1., 1., -inf],
        [1., 1., 1.]])


In [50]:
# Now implementing in A multi head attention ==> multi head + masked  ==> masked multi head attention

def multi_head_attention(embadded_dim,final_emb,num_head,masked = False):
    head_dim = embadded_dim//num_head

    if embadded_dim%num_head == 0:

        wq = nn.Linear(embadded_dim,embadded_dim)
        wk = nn.Linear(embadded_dim,embadded_dim)
        wv = nn.Linear(embadded_dim,embadded_dim)
    
        q = wq(final_emb)
        k = wk(final_emb)
        v = wv(final_emb)
    
        # since we are using multiple self attention
        batch_size,seq_len,embadded_dim = final_emb.shape
    
    
        q = q.view(batch_size,seq_len,num_head,head_dim).transpose(1,2)
        k = k.view(batch_size,seq_len,num_head,head_dim).transpose(1,2)
        v = v.view(batch_size,seq_len,num_head,head_dim).transpose(1,2)


        kT = k.transpose(-2,-1)

        scores = q@kT

        scores = scores/math.sqrt(head_dim)

        if masked:
            musked = masked_multihead_attention(seq_len)

            musked = musked.unsqueeze(0).unsqueeze(0)

            scores = scores.masked_fill(musked == 0, float('-inf'))
                                        
            

        attention = torch.softmax(scores,dim=-1)
        output = attention@v

        output = output.transpose(1,2).contiguous()

        output = output.view(batch_size,seq_len,embadded_dim)

        return output

    else:
        print("Invalida Splitting data head_dim = embadded_dim//num_head ")
        

# Cross Attention